In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retailnova;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailnova.bronze;

CREATE SCHEMA IF NOT EXISTS retailnova.silver;

CREATE SCHEMA IF NOT EXISTS retailnova.gold;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retailnova.bronze.retailnova_source;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailnova.bronze.etl_control (
    source_name STRING,
    last_processed_at TIMESTAMP
);

In [0]:
%sql
INSERT INTO retailnova.bronze.etl_control
VALUES ('customers', TIMESTAMP('1900-01-01 00:00:00'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO retailnova.bronze.etl_control
VALUES ('customers_silver', TIMESTAMP('1900-01-01 00:00:00'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT *
FROM retailnova.bronze.etl_control;

source_name,last_processed_at
customers_silver,1900-01-01T00:00:00.000Z
customers,1900-01-01T00:00:00.000Z


In [0]:
%sql

CREATE TABLE retailnova.gold.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    customer_id STRING,
    customer_name STRING,
    city STRING,
    state STRING,
    customer_tier STRING,
    updated_at TIMESTAMP,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
);

In [0]:
%sql
TRUNCATE TABLE retailnova.bronze.customers;
TRUNCATE TABLE retailnova.silver.customers;
TRUNCATE TABLE retailnova.gold.dim_customer;

In [0]:
%sql
SELECT COUNT(*) FROM retailnova.bronze.customers;


    


COUNT(*)
105000


In [0]:
%sql
SELECT COUNT(*) FROM retailnova.silver.customers;

COUNT(*)
105000


In [0]:
%sql
SELECT COUNT(*) FROM retailnova.gold.dim_customer;

COUNT(*)
105000


In [0]:
%sql

INSERT INTO retailnova.bronze.etl_control
VALUES ('products', TIMESTAMP('1900-01-01 00:00:00'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO retailnova.bronze.etl_control
VALUES ('products_silver', '1900-01-01 00:00:00');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE retailnova.bronze.etl_control
SET last_processed_at = TIMESTAMP('1900-01-01 00:00:00')
WHERE source_name = 'products';

num_affected_rows
1


In [0]:
%sql
UPDATE retailnova.bronze.etl_control
SET last_processed_at = TIMESTAMP('1900-01-01 00:00:00')
WHERE source_name = 'products_silver';

num_affected_rows
1


In [0]:
%sql

CREATE TABLE retailnova.gold.dim_product (

    product_key BIGINT GENERATED ALWAYS AS IDENTITY
        (START WITH 1 INCREMENT BY 1),

    product_id STRING,
    product_name STRING,
    category STRING,
    subcategory STRING,
    brand STRING,
    unit_price DOUBLE,
    cost_price DOUBLE,
    supplier_id STRING,
    updated_at TIMESTAMP

);

In [0]:
%sql
TRUNCATE TABLE retailnova.bronze.products;
    
TRUNCATE TABLE retailnova.silver.products;
    
TRUNCATE TABLE retailnova.gold.dim_product;
    


In [0]:
%sql
select * from retailnova.bronze.etl_control;

source_name,last_processed_at
customers_silver,2026-09-10T23:59:10.000Z
products_silver,2026-09-11T00:00:00.000Z
customers,2026-09-10T23:59:10.000Z
products,2026-09-11T00:00:00.000Z


In [0]:
%sql

CREATE TABLE retailnova.gold.dim_store (
    store_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    store_id STRING,
    store_name STRING,
    city STRING,
    state STRING,
    region STRING,
    store_type STRING,
    opening_date DATE
);

In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailnova.gold.dim_date (
    date_key INT,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    day INT,
    day_of_week INT,
    day_name STRING
);

In [0]:
%sql
DROP TABLE IF EXISTS retailnova.gold.fact_sales;
CREATE TABLE IF NOT EXISTS retailnova.gold.fact_sales (
    sales_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    order_id STRING,
    customer_key BIGINT,
    product_key BIGINT,
    store_key BIGINT,
    date_key INT,
    order_timestamp TIMESTAMP,
    quantity DOUBLE,
    unit_price DOUBLE,
    discount_amount DOUBLE,
    sales_amount DOUBLE,
    status STRING,
    payment_method STRING
);